# BÁO CÁO KHẢO SÁT & PHÂN TÍCH KHÁM PHÁ DỮ LIỆU (EDA)
## BỘ DỮ LIỆU ONLINE RETAIL II (UCI MACHINE LEARNING REPOSITORY)
**Môn học:** Kho dữ liệu và Khai phá dữ liệu - PTIT  
**Mục tiêu:** Khảo sát các bẫy dữ liệu (Data Traps), tỷ lệ thiếu dữ liệu, đơn hủy/hoàn và quy luật phân phối để thiết lập các quy tắc ETL và Data Mining.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

data_path = "../data_lake/raw/uci/online_retail_II.csv"
df = pd.read_csv(data_path)
print(f"Tổng số dòng: {len(df):,} | Số cột: {len(df.columns)}")
df.head()

### 1. Kiểm tra Kiểu Dữ liệu và Tỷ lệ Khuyết thiếu (Missing Values)
Đặc biệt chú ý đến cột `Customer ID`. Trong bài toán phân cụm khách hàng, không thể phân cụm các bản ghi thiếu mã định danh.

In [ ]:
missing_summary = pd.DataFrame({
    'Missing_Count': df.isnull().sum(),
    'Missing_Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
})
missing_summary

**Nhận xét:**
- Cột `Customer ID` thiếu xấp xỉ 20% tổng số bản ghi.
- **Quy tắc ETL:** Giữ nguyên các dòng này trong `FactSales` (gán mã khách vãng lai `GUEST_UCI`) để không làm sai lệch tổng doanh thu, nhưng **loại bỏ** khi trích xuất bảng đặc trưng RFM.

### 2. Khảo sát Hóa đơn Hủy/Hoàn (Cancellations & Negative Quantities)
Hóa đơn hủy thường có tiền tố 'C' hoặc số lượng `Quantity < 0`.

In [ ]:
cancellations = df[df['Invoice'].astype(str).str.startswith('C') | (df['Quantity'] < 0)]
print(f"Số lượng giao dịch hủy/hoàn: {len(cancellations):,} ({len(cancellations)/len(df)*100:.2f}%)")
cancellations.head()

### 3. Khảo sát Phân phối Đơn giá (Price) và Số lượng (Quantity)
Phát hiện các ngoại lệ (Outliers) như đơn giá bằng 0 hoặc các mã điều chỉnh kho (`POST`, `D`, `M`).

In [ ]:
zero_price = df[df['Price'] <= 0]
print(f"Số giao dịch có đơn giá <= 0: {len(zero_price):,}")
df[['Quantity', 'Price']].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

### 4. Kết luận Nghiệp vụ cho Thiết kế Kho Dữ liệu & ETL
1. **Bảng FactSales:** Lưu cả đơn thành công và đơn hủy (có cờ `is_cancelled = TRUE`) để tính được cả Doanh số gộp và Doanh thu thuần.
2. **Bảng loại trừ (etl_rejected_records):** Đưa các bản ghi có `Price <= 0` hoặc mã sản phẩm rác vào bảng loại trừ kèm lý do rõ ràng.
3. **Chống trùng nạp (Idempotency):** Cần sinh mã băm SHA256 dựa trên bộ `(source, invoice, stock_code, quantity, price, date)` để chạy lại pipeline nhiều lần mà không bị trùng dữ liệu.